# Practical Machine Learning — Day 02 Lab
## Data Preprocessing, Cleaning, Transformation and Pipelines
**CDAC PGCP-AI · Practical Machine Learning · Saturday 24 October 2026**

## Section 0 — Welcome and Instructions

### 🎯 Learning objectives
1. Diagnose data-quality problems and justify corrective actions.
2. Explain missingness, outliers and inconsistent records.
3. Choose suitable imputation and encoding strategies.
4. Understand how fitted preprocessing can cause data leakage.
5. Distinguish `fit`, `transform` and `fit_transform`.
6. Build numerical and categorical preprocessing **Pipelines**.
7. Combine branches with **ColumnTransformer**.
8. Apply preprocessing correctly to unseen observations.
9. Validate the transformed dataset and explain the results.

### ⏱️ Schedule (this notebook covers hours 4 to 7)
| Hour | Type | Content | Exercises |
|---|---|---|---|
| 4 | Guided self-learning | Data dictionary, investigation, decision log, concepts | SL1 – SL3 |
| 5 | **Lab** | Load, inspect, document, deterministic cleaning, features, split | A1 – A9, B1 – B2 |
| 6 | **Lab** | Pipelines, ColumnTransformer, validation, unseen data | B3 – B5, C1 – C5 |
| 7 | Guided self-learning | Compare strategies, debug, mini-challenges, conclusions | C6, D1 – D3, MC1 – MC3, C7 |

### 📁 Files (keep them in the same folder as this notebook)
- `06_titanic_data_dirty.csv` — the classroom dataset (**never edit or overwrite it**)
- `07_Data_Dictionary.xlsx` — column meanings, valid ranges and business rules

### ✍️ How the blanks work
- `____` marks a blank. Replace it with the missing method, value or expression. Each blank has a label such as `# TODO (A1-1)`.
- A cell with an unfinished blank **fails on purpose**. All other cells run as they are.
- 🔮 **Predict** boxes: write what you expect *before* running the cell. Comparing prediction and result is how you learn.
- Levels: **1** complete a method · **2** complete an argument or expression · **3** choose the right technique · **4** debug · **5** write it yourself.

### ▶️ Running cells
Shift + Enter. Run cells **in order**. If something looks strange: *Kernel → Restart & Run All* (Colab: *Runtime → Restart and run all*).

### 📤 Submit
1. This completed notebook 2. A cleaned, transformed dataset file is **not** required today; submit your notebook with **all outputs visible**, your Cleaning Decision Log and your final reflection (Section 12).

### ⚠️ Golden rules for today
- **Fix only what does not learn from data before splitting** (spaces, spelling, text-to-number, impossible values → missing). Anything that *learns* from the data (medians, means, categories, scaling) happens **after the split, inside a Pipeline, fitted on training data only**.
- Never edit the raw CSV. Flag before you delete. Explain every decision.
- Detailed feature engineering, feature selection, PCA and t-SNE are **Day 3**. No model is trained today.

---
# GUIDED SELF-LEARNING 1 (Hour 4, 60 min)
## Section 1 — Investigate the dataset on paper before you code
Open `07_Data_Dictionary.xlsx`. Use the sheets *Data Dictionary*, *Business Rules*, *Category Aliases* and *Dataset Notes*.

You may open the CSV in a text editor or Excel to look (read-only!), but do **not** change it.

### Exercise SL1 — Read the data dictionary
**Task:** Answer from the dictionary (no code):
1. Which column is the **target**? Which columns look like **identifiers**?
2. Which valid range is given for `Age`? Is an age of 0.42 valid? Why might it exist?
3. Which columns are **not** automatically good features, and why?
4. Which values are *invalid* and which are only *suspicious* for `Fare`?

<details><summary>💡 Hint (try first without it)</summary>

Read the sheet *Business Rules* slowly. The notes column explains why some unusual values are genuine.

</details>

✏️ **Your answer to SL1:** *(double-click and type here)*

&nbsp;


### Exercise SL2 — Draft your cleaning decision log (plan only)
Before you see the numbers, predict what you will do. Fill the table; you will revise it in Exercise A7 and C7.

| Problem class | What I expect to find | My planned decision | Learns from data? (Yes/No) |
|---|---|---|---|
| Spaces, capitalisation, abbreviations | | | |
| Numbers stored as text | | | |
| Impossible values (age < 0 …) | | | |
| Missing numbers | | | |
| Missing categories | | | |
| Exact duplicates | | | |
| Repeated ID, different data | | | |
| Identifier columns | | | |

### Exercise SL3 — Conceptual questions (write 1–3 sentences each)
1. Why can imputing missing ages with the median of the **whole** dataset before splitting cause leakage?
2. What is the difference between `fit`, `transform` and `fit_transform`?
3. Why do we use `handle_unknown="ignore"` in a one-hot encoder?
4. What does a `Pipeline` guarantee that separate manual steps do not?
5. What does `ColumnTransformer` do and why do we need it?
6. Why is a Fare of 512 not automatically an error, but a Fare of 5123 is?

✏️ **Your answer to SL3:** *(double-click and type here)*

&nbsp;


---
# STAGE A — Investigate and Prepare Data (Hour 5)
## Section 2 — Load and inspect

### Exercise A1: Load the CSV

**Level 1** &nbsp;|&nbsp; **Task:** Read `06_titanic_data_dirty.csv` into a DataFrame called `df`.

**You should see:** `Loaded: 06_titanic_data_dirty.csv`.

<details><summary>💡 Hint (try first without it)</summary>

Pandas has one `read_` function per file type.

</details>

In [ ]:
# ===== Exercise A1 =====
import os, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)

# Optional (Google Colab only): upload the CSV. Outside Colab this does nothing.
try:
    from google.colab import files
    if not os.path.exists("06_titanic_data_dirty.csv"):
        files.upload()
except ImportError:
    pass

DATA_FILE = "06_titanic_data_dirty.csv"
assert os.path.exists(DATA_FILE), f"{DATA_FILE} not found in {os.getcwd()}"

# Fingerprint of the raw file: at the end we prove it was never modified
RAW_FINGERPRINT = hashlib.md5(open(DATA_FILE, "rb").read()).hexdigest()

# TODO (A1-1): complete the pandas function that reads a CSV file
df = pd.____(DATA_FILE)
print("Loaded:", DATA_FILE)

### Exercise A2: Shape, types and first rows

**Level 1** &nbsp;|&nbsp; **Task:** Display the data types and non-null counts, then look at the first rows.

**You should see:** A summary of 12 columns. **Look at the Dtype of `Age` and `Fare`.**

<details><summary>💡 Hint (try first without it)</summary>

The method name is a synonym for 'information'.

</details>

**🤔 Reflect (A2):** Which columns that should be numeric are stored as text, and what might cause this?

In [ ]:
# ===== Exercise A2 =====
print("Rows, columns:", df.shape)
# TODO (A2-1): display column dtypes and non-null counts
df.____()
df.head(8)

✏️ **Your answer to A2:** *(double-click and type here)*

&nbsp;


## Section 3 — Missing values (A3)

### Exercise A3: Count missing values

**Level 1** &nbsp;|&nbsp; **Task:** Count missing cells per column and as a percentage.

**You should see:** A table sorted from most to least missing. Which column is almost empty, and what does that suggest about using it as a feature?

<details><summary>💡 Hint (try first without it)</summary>

A method that returns True for missing cells (two names exist). `True` counts as 1 in `.sum()`.

</details>

In [ ]:
# ===== Exercise A3 =====
# TODO (A3-1): True where a value is missing, then add up per column
missing = df.____().sum()
missing_report = pd.DataFrame({"missing_count": missing, "missing_percent": (missing / len(df) * 100).round(1)})
missing_report.sort_values("missing_count", ascending=False)

✏️ **Your answer to A3:** *(double-click and type here)*

&nbsp;


## Section 4 — Duplicates and repeated IDs (A4)

🔮 **Predict before you run (A4):** How many rows do you think are exact copies of an earlier row? More or fewer than 20?

✏️ *Your prediction:* &nbsp;


### Exercise A4: Exact duplicates versus repeated PassengerId

**Level 2** &nbsp;|&nbsp; **Task:** (a) Count exact duplicate rows. (b) Mark every row whose `PassengerId` appears more than once. (c) Find which repeated IDs have **conflicting** values.

**You should see:** A count, a table of repeated-ID rows and a list of conflicting IDs.

<details><summary>💡 Hint (try first without it)</summary>

(a) The method name is a synonym for 'repeated'. (b) `value_counts()` says how often each ID appears: which count means 'repeated'?

</details>

**🤔 Reflect (A4):** If the same passenger appeared in both the training set and the test set, what would happen to your test score?

In [ ]:
# ===== Exercise A4 =====
# (a) TODO (A4-1): count rows that are exact copies of an earlier row
exact_dups = df.____().sum()
print("Exact duplicate rows:", exact_dups)

# (b) how many times does each PassengerId appear?
id_counts = df["PassengerId"].value_counts()
# TODO (A4-2): keep the IDs that appear MORE than this many times (what count means "repeated"?)
repeated_ids = id_counts[id_counts > ____].index
repeated_rows = df[df["PassengerId"].isin(repeated_ids)].sort_values("PassengerId")
print("Rows whose PassengerId appears more than once:", len(repeated_rows))

# (c) provided: count distinct values per column inside each ID. 2+ distinct values = the rows disagree
distinct_per_id = df.groupby("PassengerId").nunique(dropna=False)
conflicting_ids = sorted(distinct_per_id[(distinct_per_id > 1).any(axis=1)].index)
print("IDs with CONFLICTING data:", conflicting_ids)
repeated_rows[repeated_rows["PassengerId"].isin(conflicting_ids)][["PassengerId", "Survived", "Pclass", "Sex", "Age", "Fare"]]

✏️ **Your answer to A4:** *(double-click and type here)*

&nbsp;


## Section 5 — Inconsistent categories (A5)

### Exercise A5: Inspect the categorical columns

**Level 1** &nbsp;|&nbsp; **Task:** Print the value counts (including missing) for `Sex`, `Embarked`, `Pclass` and `Survived`.

**You should see:** Several spellings of the same value, abbreviations and some unexpected values.

<details><summary>💡 Hint (try first without it)</summary>

Missing values are hidden by default (`dropna=True`). Which boolean shows them?

</details>

**🤔 Reflect (A5):** Which values need a **dictionary map**, and which should simply become missing?

In [ ]:
# ===== Exercise A5 =====
for col in ["Sex", "Embarked", "Pclass", "Survived"]:
    print(f"--- {col} ---")
    # TODO (A5-1): include the missing values in the count (dropna=True would hide them)
    print(df[col].value_counts(dropna=____))
    print()

✏️ **Your answer to A5:** *(double-click and type here)*

&nbsp;


## Section 6 — Numbers stored as text, invalid values and outliers (A6)

### Exercise A6: Convert, apply business rules and compare with the IQR rule

**Level 2** &nbsp;|&nbsp; **Task:** Convert `Age` and `Fare` to numbers; flag values that break the **business rules**; then compare with the statistical **IQR outlier rule** for `Fare`.

**You should see:** Counts of invalid ages and fares, and a much larger count of IQR outliers.

<details><summary>💡 Hint (try first without it)</summary>

`to_numeric` with `errors='coerce'`. The age limit is on the Business Rules sheet. The classic IQR fence multiplier is a number between 1 and 2.

</details>

**🤔 Reflect (A6):** Why is a statistical outlier not always an error? Which fares are genuine?

In [ ]:
# ===== Exercise A6 =====
def to_number(series):
    """Keep digits, point and minus sign; convert; unreadable text becomes NaN."""
    only_number = series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
    # TODO (A6-1): convert the text to numbers; unreadable values must become NaN
    return pd.____(only_number, errors="coerce")

age_num, fare_num = to_number(df["Age"]), to_number(df["Fare"])

# Business rules (07_Data_Dictionary.xlsx -> Business Rules)
# TODO (A6-2): the largest valid age
age_invalid = (age_num < 0) | (age_num > ____)
fare_invalid = (fare_num < 0) | (fare_num > 1000)
print("Invalid ages :", int(age_invalid.sum()))
print("Invalid fares:", int(fare_invalid.sum()))
print("Ages below 1 (infants):", int(((age_num > 0) & (age_num < 1)).sum()), "| zero fares:", int((fare_num == 0).sum()))

# Statistical outliers on the VALID fares (IQR rule)
fare_valid = fare_num[(fare_num >= 0) & (fare_num <= 1000)]
q1, q3 = fare_valid.quantile(0.25), fare_valid.quantile(0.75)
iqr = q3 - q1
# TODO (A6-3): the usual fence multiplier of the IQR rule
upper_fence = q3 + ____ * iqr
iqr_outliers = fare_valid[fare_valid > upper_fence]
print(f"IQR upper fence: {upper_fence:.1f} | valid fares above it: {len(iqr_outliers)} | highest valid fare: {fare_valid.max():.1f}")

✏️ **Your answer to A6:** *(double-click and type here)*

&nbsp;


## Section 7 — Cleaning decision log (A7)

### Exercise A7 — Cleaning plan
**Task:** Complete the table. For each problem you found in A3 – A6 write the **evidence** (a number from your outputs), the **decision**, and whether it must be done *before* or *inside the pipeline after* the split.

| Problem | Evidence (number from your output) | Decision | Before split (deterministic) or inside pipeline (learns)? | Risk |
|---|---|---|---|---|
| Mostly empty column | | | | |
| Missing Age / Fare | | | | |
| Missing Sex / Embarked / Pclass | | | | |
| Spelling variants and aliases | | | | |
| Text numbers (`22 yrs`, `£7.25`) | | | | |
| Impossible values | | | | |
| Suspicious ages 90-100 | | | | |
| Exact duplicates | | | | |
| Conflicting repeated IDs | | | | |
| Rows with no `Survived` | | | | |

<details><summary>💡 Hint (try first without it)</summary>

Use the rule of thumb: *if the decision needs a number computed from the data (median, most frequent, mean), it goes inside the pipeline.*

</details>

✏️ **Your answer to A7:** *(double-click and type here)*

&nbsp;


## Section 8 — Safe, deterministic cleaning (A8)

> **Safe** means: the step does **not** learn anything from the data. Replacing `MALE` by `male` or `-4` by missing is the same no matter which rows you look at. Computing a median **is not safe** before the split, so we leave it for the pipeline.

### Exercise A8: Clean text, numbers, duplicates, conflicts and unlabelled rows

**Level 2** &nbsp;|&nbsp; **Task:** Standardise the text columns, convert the numbers, turn impossible values into missing, remove exact duplicates and quarantine conflicting IDs and rows without a label.

**You should see:** A row-accounting table that adds up, and cleaned categories: `Sex` has male/female (+NaN), `Embarked` has S/C/Q (+NaN), `Pclass` has First/Second/Third (+NaN).

<details><summary>💡 Hint (try first without it)</summary>

strip → lower → dictionary `map` (unmapped values become NaN, provided); `drop_duplicates()` for exact copies; `np.nan` for impossible values.

</details>

**🤔 Reflect (A8):** Why do we quarantine **all** rows of a conflicting ID instead of keeping the first one?

In [ ]:
# ===== Exercise A8 =====
clean_df = df.copy()

# (1) remove the spaces around the text values
for col in ["Sex", "Embarked", "Pclass", "Survived"]:
    # TODO (A8-1): remove spaces at the start and end of every value
    clean_df[col] = clean_df[col].str.____()

# (2) make the values lower-case, then replace each by its standard label. Values not in a dictionary become NaN.
sex_map      = {"male": "male", "m": "male", "female": "female", "f": "female"}
embarked_map = {"s": "S", "southampton": "S", "c": "C", "cherbourg": "C", "q": "Q", "queenstown": "Q"}
pclass_map   = {"1": "First", "1st": "First", "2": "Second", "2nd": "Second", "3": "Third", "3rd": "Third"}
survived_map = {"0": 0, "1": 1, "no": 0, "yes": 1}
for col, mapping in {"Sex": sex_map, "Embarked": embarked_map, "Pclass": pclass_map, "Survived": survived_map}.items():
    # TODO (A8-2): write every value in lower-case (the dictionary keys are lower-case)
    clean_df[col] = clean_df[col].str.____().map(mapping)

# (3) numbers stored as text -> numbers (to_number comes from A6)
for col in ["Age", "Fare"]:
    clean_df[col] = to_number(clean_df[col])

# (4) impossible values -> missing. Rules come from the Business Rules sheet. We count what we change.
invalid_rules = {"Age": lambda s: (s < 0) | (s > 100), "Fare": lambda s: (s < 0) | (s > 1000),
                 "SibSp": lambda s: (s < 0) | (s > 10), "Parch": lambda s: (s < 0) | (s > 10)}
invalid_counts = {}
for col, rule in invalid_rules.items():
    bad = rule(clean_df[col])
    invalid_counts[col] = int(bad.sum())
    # TODO (A8-4): set the impossible values to missing (use the NumPy missing-value constant)
    clean_df.loc[bad, col] = np.____
print("Impossible values set to missing:", invalid_counts)

# (5) exact duplicates
rows_start = len(clean_df)
# TODO (A8-3): drop rows that are exact copies of an earlier row (keep the first)
clean_df = clean_df.____()
n_exact = rows_start - len(clean_df)

# (6) conflicting IDs: quarantine ALL rows of those IDs
conflict_mask = clean_df["PassengerId"].duplicated(keep=False)
quarantine_conflicts = clean_df[conflict_mask]
clean_df = clean_df[~conflict_mask]

# (7) rows without a label cannot be used for supervised learning
no_label = clean_df["Survived"].isna()
quarantine_no_label = clean_df[no_label]
clean_df = clean_df[~no_label].reset_index(drop=True)
clean_df["Survived"] = clean_df["Survived"].astype(int)

accounting = pd.DataFrame({"step": ["Raw rows", "Exact duplicates removed", "Conflicting-ID rows quarantined", "Rows without Survived quarantined", "Rows in modelling table"],
                           "rows": [len(df), n_exact, len(quarantine_conflicts), len(quarantine_no_label), len(clean_df)]})
assert len(df) == n_exact + len(quarantine_conflicts) + len(quarantine_no_label) + len(clean_df), "Row accounting does not add up"
print(accounting.to_string(index=False))
for col in ["Sex", "Embarked", "Pclass"]:
    print(clean_df[col].value_counts(dropna=False).to_dict())

✏️ **Your answer to A8:** *(double-click and type here)*

&nbsp;


## Section 9 — Choose the features (A9)

### Exercise A9: Select the candidate features

**Level 2** &nbsp;|&nbsp; **Task:** Remove the columns that should **not** be automatically used as features.

**You should see:** A table `model_df` with `Survived` plus 7 candidate features.

<details><summary>💡 Hint (try first without it)</summary>

Name the identifiers and the mostly empty column in a list and use the DataFrame method that removes columns.

</details>

**🤔 Reflect (A9):** Why would a model that uses `PassengerId` or `Ticket` be risky?

In [ ]:
# ===== Exercise A9 =====
# TODO (A9-1): list the columns to exclude: row id, free-text name, ticket text, mostly-empty cabin
exclude = ____
model_df = clean_df.drop(columns=exclude)
print("Columns kept:", list(model_df.columns))
model_df.head()

✏️ **Your answer to A9:** *(double-click and type here)*

&nbsp;


---
# STAGE B — Split Data and Build Transformations (Hours 5 and 6)
## Section 10 — Features, target and the split

### Exercise B1: Separate features and target

**Level 1** &nbsp;|&nbsp; **Task:** Create the feature table `X` and the target `y`.

**You should see:** `X` with 7 columns and `y` with only 0 and 1.

<details><summary>💡 Hint (try first without it)</summary>

`X` is everything except the target; `y` is the one column you want to predict (use its name in quotes).

</details>

In [ ]:
# ===== Exercise B1 =====
target = "Survived"
X = model_df.drop(columns=[target])
# TODO (B1-1): the target column as a Series (name of the column in quotes)
y = model_df[____]
print("X shape:", X.shape, "| y shape:", y.shape)
print("Features:", list(X.columns))

✏️ **Your answer to B1:** *(double-click and type here)*

&nbsp;


🔮 **Predict before you run (B2):** If you hold out 20% for testing, about how many rows will the test set have? Will the share of survivors be about the same in train and test if you use `stratify=y`?

✏️ *Your prediction:* &nbsp;


### Exercise B2: Split into training and test sets

**Level 2** &nbsp;|&nbsp; **Task:** Split with 20% for testing, keeping the survivor share equal in both parts, with a fixed random seed.

**You should see:** Four shapes and a table showing nearly equal survival rates in train and test.

<details><summary>💡 Hint (try first without it)</summary>

`test_size` is a fraction; `stratify` needs the target to keep class proportions.

</details>

**🤔 Reflect (B2):** Why must the split happen *before* any imputation or scaling?

In [ ]:
# ===== Exercise B2 =====
from sklearn.model_selection import train_test_split

# TODO (B2-1): hold out 20% of the rows for testing (give a fraction)
# TODO (B2-2): keep the survival proportion equal in both parts (stratify by the target)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=____, stratify=____, random_state=42)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Survival rate - train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))

✏️ **Your answer to B2:** *(double-click and type here)*

&nbsp;


## Section 11 — `fit`, `transform` and pipelines (Hour 6)

🔮 **Predict before you run (B3):** StandardScaler on `Fare` in the training set: will the scaled **test** Fare values have mean exactly 0? Why or why not?

✏️ *Your prediction:* &nbsp;


### Exercise B3: See fit and transform with your own eyes

**Level 1** &nbsp;|&nbsp; **Task:** Fit a `StandardScaler` on the training `Fare`, read what it learned, then transform the test `Fare`.

**You should see:** The learned mean of the training Fare, and scaled test values whose mean is **not exactly** 0.

<details><summary>💡 Hint (try first without it)</summary>

`fit` learns, `transform` applies. The learned mean is stored in an attribute ending with an underscore.

</details>

**🤔 Reflect (B3):** What would change if you called `fit` on the test set instead?

In [ ]:
# ===== Exercise B3 =====
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# TODO (B3-1): LEARN the mean and standard deviation from the TRAINING fares only
scaler.____(X_train[["Fare"]])

# TODO (B3-2): the attribute holding the mean learned from the training data
print("Mean learned from training Fare:", scaler.____)

# TODO (B3-3): APPLY the already-learned rule to the TEST fares (no re-learning)
test_scaled = scaler.____(X_test[["Fare"]])
print("Mean of scaled TEST Fare (missing values ignored):", float(np.nanmean(test_scaled)))
print("(By construction the mean of the scaled TRAIN Fare would be 0.)")

✏️ **Your answer to B3:** *(double-click and type here)*

&nbsp;


### Exercise B4: Numeric pipeline

**Level 3** &nbsp;|&nbsp; **Task:** Build a Pipeline for the numeric columns: fill missing values with a robust statistic, then standardise.

**You should see:** A pipeline object listing two named steps.

<details><summary>💡 Hint (try first without it)</summary>

Which central statistic is robust to extreme fares? And which class standardises?

</details>

**🤔 Reflect (B4):** Why does the order impute-then-scale matter?

In [ ]:
# ===== Exercise B4 =====
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]

numeric_pipeline = Pipeline(steps=[
    # TODO (B4-1): fill missing numbers with the median (choose the strategy name)
    ("impute", SimpleImputer(strategy="____")),
    # TODO (B4-2): choose the scaler that gives mean 0 and standard deviation 1
    ("scale", ____()),
])
numeric_pipeline

✏️ **Your answer to B4:** *(double-click and type here)*

&nbsp;


### Exercise B5: Categorical pipeline

**Level 3** &nbsp;|&nbsp; **Task:** Build a Pipeline for the categorical columns: fill missing categories with the most frequent value, then one-hot encode — without crashing on unseen categories.

**You should see:** A pipeline object listing two named steps.

<details><summary>💡 Hint (try first without it)</summary>

Strategy name: the *most frequent* category. For unseen categories the encoder needs `handle_unknown` set to the word that means 'do not raise an error'.

</details>

**🤔 Reflect (B5):** What will the encoder output for a category it has never seen?

In [ ]:
# ===== Exercise B5 =====
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(steps=[
    # TODO (B5-1): fill missing categories with the most frequent value (strategy name)
    ("impute", SimpleImputer(strategy="____")),
    # TODO (B5-2): do NOT crash when a category appears that was not seen in training
    ("onehot", OneHotEncoder(handle_unknown="____", sparse_output=False)),
])
categorical_pipeline

✏️ **Your answer to B5:** *(double-click and type here)*

&nbsp;


---
# STAGE C — Combine, Validate and Interpret (Hour 6)
## Section 12 — ColumnTransformer

### Exercise C1: Combine the branches

**Level 3** &nbsp;|&nbsp; **Task:** Combine the numeric and categorical pipelines in one `ColumnTransformer`.

**You should see:** A diagram-like object with two branches: `num` and `cat`.

<details><summary>💡 Hint (try first without it)</summary>

The class name is exactly what it does: it *transforms columns*, a different transformer for each group.

</details>

**🤔 Reflect (C1):** What would happen to a column you do not list? (check the `remainder` argument)

In [ ]:
# ===== Exercise C1 =====
import sklearn.compose as skc

# TODO (C1-1): the class of this module that applies different pipelines to different groups of columns
preprocess = skc.____(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
], remainder="drop")
preprocess

✏️ **Your answer to C1:** *(double-click and type here)*

&nbsp;


🔮 **Predict before you run (C2):** How many columns will the transformed training matrix have? (4 numeric + the one-hot columns). Will train and test have the same number of columns?

✏️ *Your prediction:* &nbsp;


### Exercise C2: Fit on training data, transform both

**Level 1** &nbsp;|&nbsp; **Task:** Fit the preprocessing on `X_train` and transform `X_train` and `X_test`.

**You should see:** Two matrices with the **same number of columns** and different numbers of rows.

<details><summary>💡 Hint (try first without it)</summary>

Training: `fit` and `transform` in one call. Test: only `transform`.

</details>

**🤔 Reflect (C2):** Why is calling `fit_transform` on the test set a mistake?

In [ ]:
# ===== Exercise C2 =====
# TODO (C2-1): LEARN from the training data and transform it in one call
X_train_prep = preprocess.____(X_train)
# TODO (C2-2): APPLY the learned rules to the test data (no learning!)
X_test_prep = preprocess.____(X_test)

print("Train matrix:", X_train_prep.shape, "| Test matrix:", X_test_prep.shape)
imputer = preprocess.named_transformers_["num"].named_steps["impute"]
print("Medians learned from TRAIN:", dict(zip(numeric_features, imputer.statistics_.round(2).tolist())))

✏️ **Your answer to C2:** *(double-click and type here)*

&nbsp;


### Exercise C3: Feature names and dimensions

**Level 1** &nbsp;|&nbsp; **Task:** Turn the matrix back into a labelled table.

**You should see:** A table whose columns are named like `num__Age`, `cat__Sex_male`.

<details><summary>💡 Hint (try first without it)</summary>

A method of the fitted ColumnTransformer that returns the output column names; its name starts with `get_feature_names_out`.

</details>

**🤔 Reflect (C3):** Which input column produced the most output columns, and why?

In [ ]:
# ===== Exercise C3 =====
# TODO (C3-1): ask the fitted preprocessor for the names of the output columns
feature_names = preprocess.____()
X_train_df = pd.DataFrame(X_train_prep, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_prep, columns=feature_names, index=X_test.index)
print("Output columns:", len(feature_names))
print(list(feature_names))
X_train_df.head()

✏️ **Your answer to C3:** *(double-click and type here)*

&nbsp;


### Exercise C4: Validate the transformed data

**Level 5** &nbsp;|&nbsp; **Task:** Run the provided checks, then **write three checks yourself**.

**You should see:** All provided checks pass, then your own pass too.

<details><summary>💡 Hint (try first without it)</summary>

Ideas: same width; no NaN anywhere; one-hot columns for one feature add up to 1 in every row (e.g. the `cat__Sex_` columns).

</details>

**🤔 Reflect (C4):** Why is it correct that test numeric means are not exactly 0?

In [ ]:
# ===== Exercise C4 (provided checks) =====
assert X_train_df.shape[1] == X_test_df.shape[1], "Train and test have different columns"
assert X_train_df.index.is_unique and X_test_df.index.is_unique
assert set(X_train_df.index).isdisjoint(set(X_test_df.index)), "Same row in train and test!"
num_cols = [c for c in X_train_df.columns if c.startswith("num__")]
assert np.allclose(X_train_df[num_cols].mean(), 0, atol=1e-6), "Train numeric means should be ~0"
assert np.allclose(X_train_df[num_cols].std(ddof=0), 1, atol=1e-6), "Train numeric std should be ~1"
print("Provided checks passed")
print("Test numeric means (small but not exactly 0):")
print(X_test_df[num_cols].mean().round(3).to_dict())

In [ ]:
# ===== Exercise C4-L5 (Level 5: write it yourself) =====
# TODO (C4-L5): write THREE assert statements with helpful messages:
#   1) neither X_train_df nor X_test_df contains any missing value
#   2) every one-hot "cat__Sex_" group adds up to exactly 1 in every row (train and test)
#   3) the number of rows in each matrix matches the number of rows in X_train / X_test

# YOUR CODE HERE


✏️ **Your answer to C4:** *(double-click and type here)*

&nbsp;


🔮 **Predict before you run (C5):** A new passenger arrives: Age missing, Embarked written as the new value 'X'. What will the age become after preprocessing, and what will the three Embarked columns be?

✏️ *Your prediction:* &nbsp;


### Exercise C5: Transform an unseen passenger

**Level 1** &nbsp;|&nbsp; **Task:** Run a brand-new passenger through the **already fitted** preprocessing.

**You should see:** One row with the median age (scaled), and zeros in all Embarked columns.

<details><summary>💡 Hint (try first without it)</summary>

The new row needs only `transform`; the fitted preprocessing remembers everything.

</details>

**🤔 Reflect (C5):** What would have happened without `handle_unknown='ignore'`?

In [ ]:
# ===== Exercise C5 =====
new_passenger = pd.DataFrame([{"Pclass": "Second", "Sex": "female", "Age": np.nan, "SibSp": 1, "Parch": 0, "Fare": 26.0, "Embarked": "X"}])
# TODO (C5-1): apply the fitted preprocessing to the new passenger (no re-learning)
new_prep = preprocess.____(new_passenger)
new_df = pd.DataFrame(new_prep, columns=feature_names)
print(new_df.T.round(3).to_string())

✏️ **Your answer to C5:** *(double-click and type here)*

&nbsp;


---
# GUIDED SELF-LEARNING 2 (Hour 7, 60 min)
## Section 13 — Compare strategies, debug, mini-challenges, conclusions
Hour 7 is independent work: finish any unfinished exercises first, then work through the tasks below.

### Exercise C6 — Compare two preprocessing strategies
**Task (Level 5, write it yourself):** Build a second numeric pipeline that fills missing values with the **mean** instead of the median. Fit both versions on `X_train` and compare, for `Age` and `Fare`: (a) the value used to fill missing numbers, (b) the transformed value of a passenger with a missing Fare. Which strategy would you defend, and why?

<details><summary>💡 Hint (try first without it)</summary>

Copy numeric_pipeline, change the strategy, wrap each in `skc.ColumnTransformer` (imported in C1) for numeric_features only, call `fit` on X_train. `imputer.statistics_` shows the learned fill values.

</details>

In [ ]:
# ===== Exercise C6 =====
# YOUR CODE HERE


✏️ **Your answer to C6:** *(double-click and type here)*

&nbsp;


### Exercises D1 – D3 — Debugging (Level 4)
Each cell below has a bug. Predict what is wrong, then fix it so the check at the end passes.

**D1 — Leakage.** The learner fits the preprocessing on the **test** data. Fix it.

In [ ]:
# ===== Exercise D1 (Level 4: DEBUG) =====
from sklearn.base import clone
bad_preprocess = clone(preprocess)
bad_train = bad_preprocess.fit_transform(X_train)
bad_test = bad_preprocess.fit_transform(X_test)        # <- what is wrong here?

# the check: the TEST medians used must equal the TRAIN medians
used = bad_preprocess.named_transformers_["num"].named_steps["impute"].statistics_
train_medians = preprocess.named_transformers_["num"].named_steps["impute"].statistics_
assert np.allclose(used, train_medians), "The preprocessing was re-learned from the test data (leakage)!"

**D2 — Crash on an unseen category.** This encoder works on training data but crashes on new data. Fix the encoder.

In [ ]:
# ===== Exercise D2 (Level 4: DEBUG) =====
strict_encoder = OneHotEncoder(sparse_output=False)
strict_encoder.fit(X_train[["Embarked"]].fillna("S"))
new_ports = pd.DataFrame({"Embarked": ["S", "C", "X"]})
print(strict_encoder.transform(new_ports))       # <- ValueError: unknown category "X"

**D3 — Imputing before splitting.** This cell fills missing fares with the **mean of the whole modelling table**. Show that this uses test information, and replace it by the correct approach (a fitted imputer).

In [ ]:
# ===== Exercise D3 (Level 4: DEBUG) =====
whole_mean = model_df["Fare"].mean()             # <- computed on ALL rows, including the future test rows
train_mean = X_train["Fare"].mean()
print("Mean fare of all rows:", round(whole_mean, 4), "| mean fare of training rows:", round(train_mean, 4))
assert whole_mean == train_mean, "The fill value depends on rows that belong to the test set: learn it from the training rows only"

✏️ **Your answer to D1-D3:** *(double-click and type here)*

&nbsp;


### Mini-challenges (MC1 – MC3, Level 5)
**MC1 — Effect of extreme fares.** Compare the **standard deviation learned by StandardScaler** for `Fare` in the training set with and without the 5 highest fares. How much does it change and what does that tell you about outliers and scaling?

**MC2 — Different strategy per column.** Build a pipeline where `Age` is filled with the **mean** and `Fare` with the **median** (two separate numeric branches). What are the two learned fill values?

**MC3 — Pipeline on quarantined rows.** Apply the fitted `preprocess` to the conflicting-ID rows you quarantined in A8 (after dropping the columns not used as features). Report the shape and whether any value is missing. Why is this an acceptable use of `transform`?

In [ ]:
# ===== Mini-Challenges (write your code below) =====
# YOUR CODE HERE


✏️ **Your answer to MC1-MC3:** *(double-click and type here)*

&nbsp;


### Exercise C7 — Final decision log, raw-file check and reflection
**Part 1.** Complete the final Cleaning Decision Log (revise your plan from SL2 and A7 using what you learned).

| Problem | Final decision | Reason | Done before split or inside pipeline? | Remaining risk |
|---|---|---|---|---|
| Spelling, case, aliases | | | | |
| Text numbers | | | | |
| Impossible values | | | | |
| Missing numbers | | | | |
| Missing categories | | | | |
| Exact duplicates | | | | |
| Conflicting IDs | | | | |
| Identifier / mostly empty columns | | | | |
| Outliers kept (genuine fares, ages 90-100) | | | | |

**Part 2.** Write your final reflection (2–4 sentences each): *What did you diagnose? What did you decide and why? Where could leakage have happened and how did the pipeline prevent it? What remains unresolved? What will Day 3 build on?*

✏️ **Your decision log and reflection:** *(double-click and write here)*

&nbsp;

&nbsp;


In [ ]:
# ===== Raw file check (provided) =====
import hashlib
assert hashlib.md5(open(DATA_FILE, "rb").read()).hexdigest() == RAW_FINGERPRINT, "Raw file was modified!"
print("Raw file untouched")

---
## ✅ Submission checklist
- [ ] All blanks completed; *Restart & Run All* works with outputs visible
- [ ] Prediction boxes, SL1–SL3, A7 table, C6, D1–D3, MC1–MC3 and C7 completed
- [ ] Final decision log and reflection written
- [ ] Raw CSV unchanged

**See you on Day 3: feature engineering, feature selection, PCA and t-SNE.** 🎓